# Station Stacking v5 - KMIA

Wide HRRR/GFS same-day 11am notebook for `KMIA`.

This version uses the same feature engineering as v4, but tunes Optuna trials against MAE instead of RMSE. Artifacts are written to `data/calibration/station_stacking_v5`.


In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

STATION_ID = "KMIA"
FAST_MODE = False
OPTUNA_TRIALS = 100
STACK_OPTUNA_TRIALS = 50
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
from src.calibration.station_stacking import (
    StationStackingConfig,
    missing_model_dependencies,
    run_station_year_split_experiment,
)


## V5 Feature Engineering

Same feature set as v4: v2/v3 temperature features plus forecast and observed precipitation signals. Forecast precipitation comes from provider-prefixed SDK summary columns such as `gfs_forecast_precip_total_mm`, with `gfs_precip_amount` as a fallback for older caches.


In [3]:
import numpy as np
import pandas as pd
import src.calibration.station_stacking as station_stacking_module

V5_PROVIDERS = ("gfs", "hrrr")
V5_FEATURE_COLUMNS = [
    "v2_recent_heat_anomaly_f",
    "v2_recent_heat_momentum_f",
    "v2_morning_warmup_to_consensus_f",
    "v2_consensus_minus_7d_actual_f",
    "v2_spread_per_warmup_f",
    "v2_humidity_warmup_interaction",
    "v3_high_so_far_above_current_f",
    "v3_remaining_warmup_from_high_so_far_f",
    "v3_high_so_far_minus_lag_1d_f",
    "v3_high_so_far_minus_7d_actual_f",
    "v3_remaining_warmup_per_spread_f",
    "v3_humidity_remaining_warmup_interaction",
    "v4_forecast_precip_total_mean_mm",
    "v4_forecast_precip_total_max_mm",
    "v4_forecast_precip_total_spread_mm",
    "v4_forecast_precip_max_1h_mean_mm",
    "v4_forecast_precip_hours_mean",
    "v4_forecast_precip_intensity_mean",
    "v4_forecast_precip_intensity_max",
    "v4_any_forecast_precip",
    "v4_all_forecast_precip",
    "v4_observed_precip_any",
    "v4_observed_precip_recent_mm_est",
    "v4_forecast_total_minus_observed_recent_mm",
    "v4_forecast_observed_precip_match",
    "v4_forecast_wet_observed_dry",
    "v4_observed_wet_forecast_dry",
    "v4_precip_humidity_interaction",
    "v4_precip_remaining_warmup_interaction",
]


def _num(frame: pd.DataFrame, column: str) -> pd.Series:
    if column in frame:
        return pd.to_numeric(frame[column], errors="coerce")
    return pd.Series(np.nan, index=frame.index, dtype="float64")


def _bool_num(frame: pd.DataFrame, column: str) -> pd.Series:
    if column not in frame:
        return pd.Series(0, index=frame.index, dtype="int64")
    series = frame[column]
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0).gt(0).astype(int)
    return series.astype("string").str.lower().isin({"1", "true", "yes", "y"}).astype(int)


def _provider_series(frame: pd.DataFrame, provider: str, primary: str, fallback: str | None = None) -> pd.Series:
    primary_column = f"{provider}_{primary}"
    if primary_column in frame:
        return _num(frame, primary_column)
    if fallback is not None:
        return _num(frame, f"{provider}_{fallback}")
    return pd.Series(np.nan, index=frame.index, dtype="float64")


def _provider_matrix(frame: pd.DataFrame, primary: str, fallback: str | None = None) -> pd.DataFrame:
    return pd.DataFrame(
        {provider: _provider_series(frame, provider, primary, fallback) for provider in V5_PROVIDERS},
        index=frame.index,
    )


def add_v5_feature_engineering(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    observed_temp = _num(out, "observed_temp_at_as_of_f")
    high_so_far = _num(out, "observed_high_temp_through_as_of_f")
    observed_humidity = _num(out, "observed_humidity_at_as_of")
    provider_mean = _num(out, "provider_mean_high_f")
    provider_spread = _num(out, "provider_spread_high_f")
    lag_1d = _num(out, "actual_high_lag_1d")
    roll_7d = _num(out, "actual_high_roll_7d_mean")
    roll_30d = _num(out, "actual_high_roll_30d_mean")

    warmup_to_consensus = provider_mean - observed_temp
    remaining_warmup = provider_mean - high_so_far
    out["v2_recent_heat_anomaly_f"] = lag_1d - roll_30d
    out["v2_recent_heat_momentum_f"] = roll_7d - roll_30d
    out["v2_morning_warmup_to_consensus_f"] = warmup_to_consensus
    out["v2_consensus_minus_7d_actual_f"] = provider_mean - roll_7d
    out["v2_spread_per_warmup_f"] = provider_spread / warmup_to_consensus.abs().clip(lower=1.0)
    out["v2_humidity_warmup_interaction"] = (observed_humidity / 100.0) * warmup_to_consensus
    out["v3_high_so_far_above_current_f"] = high_so_far - observed_temp
    out["v3_remaining_warmup_from_high_so_far_f"] = remaining_warmup
    out["v3_high_so_far_minus_lag_1d_f"] = high_so_far - lag_1d
    out["v3_high_so_far_minus_7d_actual_f"] = high_so_far - roll_7d
    out["v3_remaining_warmup_per_spread_f"] = remaining_warmup / provider_spread.abs().clip(lower=1.0)
    out["v3_humidity_remaining_warmup_interaction"] = (observed_humidity / 100.0) * remaining_warmup

    precip_total = _provider_matrix(out, "forecast_precip_total_mm", fallback="precip_amount")
    precip_max_1h = _provider_matrix(out, "forecast_precip_max_1h_mm")
    precip_hours = _provider_matrix(out, "forecast_precip_hours_count")
    precip_intensity = _provider_matrix(out, "forecast_precip_intensity_code")
    has_precip = _provider_matrix(out, "forecast_has_precip").fillna(0).clip(lower=0, upper=1)

    out["v4_forecast_precip_total_mean_mm"] = precip_total.mean(axis=1)
    out["v4_forecast_precip_total_max_mm"] = precip_total.max(axis=1)
    out["v4_forecast_precip_total_spread_mm"] = precip_total.max(axis=1) - precip_total.min(axis=1)
    out["v4_forecast_precip_max_1h_mean_mm"] = precip_max_1h.mean(axis=1)
    out["v4_forecast_precip_hours_mean"] = precip_hours.mean(axis=1)
    out["v4_forecast_precip_intensity_mean"] = precip_intensity.mean(axis=1)
    out["v4_forecast_precip_intensity_max"] = precip_intensity.max(axis=1)
    out["v4_any_forecast_precip"] = has_precip.max(axis=1).fillna(0).astype(int)
    out["v4_all_forecast_precip"] = has_precip.min(axis=1).fillna(0).astype(int)

    observed_any = (
        _bool_num(out, "observed_is_raining_at_as_of")
        | _bool_num(out, "observed_is_drizzle_at_as_of")
        | _bool_num(out, "observed_is_snowing_at_as_of")
    ).astype(int)
    observed_recent_mm = _num(out, "observed_precip_recent_at_as_of") * 25.4
    out["v4_observed_precip_any"] = observed_any
    out["v4_observed_precip_recent_mm_est"] = observed_recent_mm
    out["v4_forecast_total_minus_observed_recent_mm"] = out["v4_forecast_precip_total_mean_mm"] - observed_recent_mm
    out["v4_forecast_observed_precip_match"] = out["v4_any_forecast_precip"].eq(observed_any).astype(int)
    out["v4_forecast_wet_observed_dry"] = (out["v4_any_forecast_precip"].eq(1) & observed_any.eq(0)).astype(int)
    out["v4_observed_wet_forecast_dry"] = (observed_any.eq(1) & out["v4_any_forecast_precip"].eq(0)).astype(int)
    out["v4_precip_humidity_interaction"] = out["v4_forecast_precip_total_mean_mm"] * (observed_humidity / 100.0)
    out["v4_precip_remaining_warmup_interaction"] = out["v4_forecast_precip_total_mean_mm"] * remaining_warmup
    return out


if not hasattr(station_stacking_module, "_v5_original_build_station_wide_dataset"):
    station_stacking_module._v5_original_build_station_wide_dataset = station_stacking_module.build_station_wide_dataset


def build_station_wide_dataset_v5(*args, **kwargs):
    frame = station_stacking_module._v5_original_build_station_wide_dataset(*args, **kwargs)
    return add_v5_feature_engineering(frame)


station_stacking_module.build_station_wide_dataset = build_station_wide_dataset_v5
V5_FEATURE_COLUMNS


['v2_recent_heat_anomaly_f',
 'v2_recent_heat_momentum_f',
 'v2_morning_warmup_to_consensus_f',
 'v2_consensus_minus_7d_actual_f',
 'v2_spread_per_warmup_f',
 'v2_humidity_warmup_interaction',
 'v3_high_so_far_above_current_f',
 'v3_remaining_warmup_from_high_so_far_f',
 'v3_high_so_far_minus_lag_1d_f',
 'v3_high_so_far_minus_7d_actual_f',
 'v3_remaining_warmup_per_spread_f',
 'v3_humidity_remaining_warmup_interaction',
 'v4_forecast_precip_total_mean_mm',
 'v4_forecast_precip_total_max_mm',
 'v4_forecast_precip_total_spread_mm',
 'v4_forecast_precip_max_1h_mean_mm',
 'v4_forecast_precip_hours_mean',
 'v4_forecast_precip_intensity_mean',
 'v4_forecast_precip_intensity_max',
 'v4_any_forecast_precip',
 'v4_all_forecast_precip',
 'v4_observed_precip_any',
 'v4_observed_precip_recent_mm_est',
 'v4_forecast_total_minus_observed_recent_mm',
 'v4_forecast_observed_precip_match',
 'v4_forecast_wet_observed_dry',
 'v4_observed_wet_forecast_dry',
 'v4_precip_humidity_interaction',
 'v4_precip_r

## Model Scores


In [4]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    output_dir=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5",
)
result = run_station_year_split_experiment(config)
result.scoreboard


[I 2026-06-10 11:32:45,150] A new study created in memory with name: no-name-2829b4ce-f579-4a7e-971b-443b525a3b46
[I 2026-06-10 11:32:56,230] Trial 0 finished with value: 1.2757928979179471 and parameters: {'n_estimators': 799, 'learning_rate': 0.12369619597856178, 'max_depth': 6, 'min_child_weight': 2.385234757844707, 'gamma': 0.7800932022121826, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.6677615511747083}. Best is trial 0 with value: 1.2757928979179471.
[I 2026-06-10 11:34:06,576] Trial 1 finished with value: 1.1920083264975587 and parameters: {'n_estimators': 1440, 'learning_rate': 0.0032515743808034223, 'max_depth': 8, 'min_child_weight': 8.23143373099555, 'gamma': 1.0616955533913808, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.2922905212920093}. Best is trial 1 with value: 1.1920083264975587.
[I 2026-06-10 11:34:43,220] Tri

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,562,1.099699,1.603522
1,validation_2024_2025,lightgbm,562,1.143334,1.692048
2,validation_2024_2025,catboost,562,1.091558,1.575600
3,validation_2024_2025,hrrr_raw,562,2.070740,2.738868
4,validation_2024_2025,gfs_raw,562,3.592958,4.064970
5,test_2026,xgboost,104,1.610924,2.263734
6,test_2026,lightgbm,104,1.627669,2.385838
7,test_2026,catboost,104,1.550583,2.063820
8,test_2026,ridge_stack,104,1.552819,2.095250
9,test_2026,hrrr_raw,104,2.488269,3.149804


## Rounded Within 1F Accuracy


In [5]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="test_2026"),
    ],
    ignore_index=True,
)

predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
3,test_2026,lightgbm,104,64,61.538462
0,test_2026,catboost,104,61,58.653846
5,test_2026,xgboost,104,61,58.653846
4,test_2026,ridge_stack,104,60,57.692308
2,test_2026,hrrr_raw,104,35,33.653846
1,test_2026,gfs_raw,104,8,7.692308
10,validation_2024_2025,xgboost,562,430,76.512456
6,validation_2024_2025,catboost,562,428,76.156584
9,validation_2024_2025,lightgbm,562,415,73.843416
8,validation_2024_2025,hrrr_raw,562,266,47.330961


## Precipitation Feature Coverage


In [6]:
precip_columns = [column for column in result.features.columns if "precip" in column.lower()]
coverage = (
    result.features[precip_columns]
    .notna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("coverage_pct")
    .reset_index()
    .rename(columns={"index": "feature"})
)
coverage


,feature,coverage_pct
0,observed_precip_intensity_code,100.000000
1,v4_forecast_observed_precip_match,100.000000
2,v4_observed_precip_any,100.000000
3,v4_all_forecast_precip,100.000000
4,v4_any_forecast_precip,100.000000
5,observed_precip_intensity,100.000000
6,v4_forecast_precip_total_mean_mm,99.745418
7,v4_forecast_precip_total_max_mm,99.745418
8,v4_forecast_precip_total_spread_mm,99.745418
9,v4_precip_remaining_warmup_interaction,99.745418


## Version Comparison


In [7]:
comparison_frames = []
for version, folder in [
    ("v1", PROJECT_ROOT / "data" / "calibration" / "station_stacking"),
    ("v2", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v2"),
    ("v3", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v3"),
    ("v4", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v4"),
    ("v5", PROJECT_ROOT / "data" / "calibration" / "station_stacking_v5"),
]:
    path = folder / f"{STATION_ID}_year_split_scoreboard.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["version"] = version
        comparison_frames.append(frame)

version_comparison = pd.concat(comparison_frames, ignore_index=True) if comparison_frames else pd.DataFrame()
if not version_comparison.empty:
    version_comparison = version_comparison.sort_values(["period", "mae_f", "version", "method"]).reset_index(drop=True)
version_comparison


,period,method,count,mae_f,rmse_f,version
0,test_2026,catboost,104,1.550583,2.063820,v5
1,test_2026,ridge_stack,104,1.552819,2.095250,v5
2,test_2026,xgboost,104,1.610924,2.263734,v5
3,test_2026,lightgbm,104,1.627669,2.385838,v5
4,test_2026,ridge_stack,133,1.840590,2.620895,v3
5,test_2026,ridge_stack,133,1.842646,2.629061,v2
6,test_2026,lightgbm,133,1.911371,2.766686,v2
7,test_2026,catboost,133,1.912569,2.720325,v3
8,test_2026,catboost,133,1.916566,2.701904,v2
9,test_2026,lightgbm,133,1.939547,2.785576,v3


## 2026 Weather Brackets


In [8]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,104,1.610924,2.263734,44.230769
1,lightgbm,104,1.627669,2.385838,39.423077
2,catboost,104,1.550583,2.063820,40.384615
3,ridge_stack,104,1.552819,2.095250,40.384615
4,hrrr_raw,104,2.488269,3.149804,23.076923
5,gfs_raw,104,4.321691,4.739521,1.923077


In [9]:
import pandas as pd


def adjacent_brackets(bracket):
    if pd.isna(bracket):
        return []
    text = str(bracket).strip()
    if not text or "-" not in text:
        return []
    try:
        lower = int(text.split("-", 1)[0])
    except ValueError:
        return []
    return [
        f"{lower - 2}-{lower - 1}",
        f"{lower}-{lower + 1}",
        f"{lower + 2}-{lower + 3}",
    ]


bracket_3way = result.bracket_predictions.copy()

valid = bracket_3way["actual_bracket"].notna() & bracket_3way["predicted_bracket"].astype(str).str.strip().ne("")
bracket_3way = bracket_3way.loc[valid].copy()
bracket_3way["picked_brackets"] = bracket_3way["predicted_bracket"].map(adjacent_brackets)
bracket_3way["three_bracket_hit"] = bracket_3way.apply(
    lambda row: row["actual_bracket"] in row["picked_brackets"],
    axis=1,
)

three_bracket_accuracy = (
    bracket_3way
    .groupby("method", as_index=False)
    .agg(
        count=("three_bracket_hit", "size"),
        exact_bracket_accuracy_pct=("bracket_hit", lambda x: x.mean() * 100),
        three_bracket_accuracy_pct=("three_bracket_hit", lambda x: x.mean() * 100),
    )
    .sort_values("three_bracket_accuracy_pct", ascending=False)
)

three_bracket_accuracy


,method,count,exact_bracket_accuracy_pct,three_bracket_accuracy_pct
4,ridge_stack,104,40.384615,87.500000
5,xgboost,104,44.230769,87.500000
3,lightgbm,104,39.423077,86.538462
0,catboost,104,40.384615,85.576923
2,hrrr_raw,104,23.076923,68.269231
1,gfs_raw,104,1.923077,27.884615
